In [25]:
import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

for d in ['/kaggle/input/datathon-2026-round-1/','datathon-2026-round-1/','../datathon-2026-round-1/',r'E:\\ThiKHDL\\Datathon_KHDL5\\datathon-2026-round-1\\\\']:
    if os.path.exists(d): DATA_DIR=d; break

sales = pd.read_csv(DATA_DIR+'sales.csv', parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
sub_form = pd.read_csv(DATA_DIR+'sample_submission.csv', parse_dates=['Date'])

sales['year'] = sales['Date'].dt.year
sales['month'] = sales['Date'].dt.month
sales['day'] = sales['Date'].dt.day
sales['dow'] = sales['Date'].dt.dayofweek

train = sales[sales['year'] < 2022].copy()
val = sales[sales['year'] == 2022].copy()
annual = sales.groupby('year')[['Revenue','COGS']].sum()

In [26]:
def compute_metrics(yt, yp):
    mae = np.mean(np.abs(yt - yp))
    rmse = np.sqrt(np.mean((yt - yp)**2))
    r2 = 1 - np.sum((yt-yp)**2) / np.sum((yt-yt.mean())**2)
    return mae, rmse, r2

report_rows = []

In [27]:
prev = sales[sales['year']==2021][['month','day','Revenue']].rename(columns={'Revenue':'sn_r'})
v = val.merge(prev, on=['month','day'], how='left').fillna(method='ffill')
m = compute_metrics(val['Revenue'].values, v['sn_r'].values)
report_rows.append({'Mô hình': 'Seasonal naive', 'MAE': m[0], 'RMSE': m[1], 'R2': m[2], 'Ghi chú': 'Baseline theo mùa vụ'})

In [28]:
roll_val = np.full(len(val), train['Revenue'].tail(90).mean())
m = compute_metrics(val['Revenue'].values, roll_val)
report_rows.append({'Mô hình': 'Rolling average', 'MAE': m[0], 'RMSE': m[1], 'R2': m[2], 'Ghi chú': 'Baseline ổn định'})

In [29]:
ann_r = sales.groupby('year')['Revenue'].transform('mean')
ann_c = sales.groupby('year')['COGS'].transform('mean')
sales['rn'] = sales['Revenue']/ann_r
sales['cn'] = sales['COGS']/ann_c
sea = (sales.groupby(['month','day'])[['rn','cn']].mean().reset_index().rename(columns={'rn':'sea_rev','cn':'sea_cogs'}))

base_rev = annual.loc[2022,'Revenue']/365
base_cogs = annual.loc[2022,'COGS']/365
GR = (annual.loc[2022,'Revenue']/annual.loc[2020,'Revenue'])**(1/2)
GC = (annual.loc[2022,'COGS']/annual.loc[2020,'COGS'])**(1/2)

def predict_baseline(df, ar, ac, gr, gc, sea_df, base_yr=2022):
    d=df.merge(sea_df,on=['month','day'],how='left')
    d['sea_rev']=d['sea_rev'].fillna(1.0); d['sea_cogs']=d['sea_cogs'].fillna(1.0)
    d['ya']=d['year']-base_yr
    return (ar*gr**d['ya']*d['sea_rev']).values, (ac*gc**d['ya']*d['sea_cogs']).values

def monthly_cv_bias():
    res=[]
    for tys,vy,ay,py in [([2019,2020],2021,2020,2019),([2019,2020,2021],2022,2021,2019)]:
        tr=sales[sales['year'].isin(tys)].copy()
        a2=tr.groupby('year')['Revenue'].transform('mean'); b2=tr.groupby('year')['COGS'].transform('mean')
        tr['r2']=tr['Revenue']/a2; tr['c2']=tr['COGS']/b2
        st=tr.groupby(['month','day'])[['r2','c2']].mean().reset_index().rename(columns={'r2':'sea_rev','c2':'sea_cogs'})
        ar=annual.loc[ay,'Revenue']/365; ac=annual.loc[ay,'COGS']/365
        n=max(ay-py,1)
        gr2=(annual.loc[ay,'Revenue']/annual.loc[py,'Revenue'])**(1/n)
        gc2=(annual.loc[ay,'COGS']/annual.loc[py,'COGS'])**(1/n)
        vv=sales[sales['year']==vy].copy()
        pp,pc=predict_baseline(vv,ar,ac,gr2,gc2,st,ay)
        vv['pr']=pp; vv['pc']=pc
        br=vv.groupby('month').apply(lambda g:g['Revenue'].mean()/g['pr'].mean())
        bc=vv.groupby('month').apply(lambda g:g['COGS'].mean()/g['pc'].mean())
        res.append((br,bc))
    br_avg=((res[0][0]+res[1][0])/2).reset_index(); bc_avg=((res[0][1]+res[1][1])/2).reset_index()
    br_avg.columns=['month','bm_r']; bc_avg.columns=['month','bm_c']
    return br_avg.merge(bc_avg,on='month')

month_bias = monthly_cv_bias()

post=sales[sales['year'].between(2019,2022)].copy()
pp,pc=predict_baseline(post,base_rev,base_cogs,GR,GC,sea)
post['pr']=pp; post['pc']=pc
post=post.merge(month_bias,on='month',how='left')
post['pm']=post['pr']*post['bm_r']; post['cm']=post['pc']*post['bm_c']
dr=post.groupby('dow').apply(lambda g:g['Revenue'].mean()/g['pm'].mean())
dc=post.groupby('dow').apply(lambda g:g['COGS'].mean()/g['cm'].mean())
dow_bias=pd.DataFrame({'dow':dr.index,'bd_r':dr.values,'bd_c':dc.values})

def apply_stat_baseline(df):
    pr,pc=predict_baseline(df,base_rev,base_cogs,GR,GC,sea)
    out=df[['Date','year','month','day','dow']].copy()
    out['pr']=pr; out['pc']=pc
    out=out.merge(month_bias,on='month',how='left')
    out['pm']=out['pr']*out['bm_r']; out['cm']=out['pc']*out['bm_c']
    out=out.merge(dow_bias,on='dow',how='left')
    out['pmd']=(out['pm']*out['bd_r']).clip(lower=0)
    out['cmd']=(out['cm']*out['bd_c']).clip(lower=0,upper=out['pmd']*1.1)
    return out

bv=apply_stat_baseline(val)

In [30]:
def make_features(df):
    d=df.copy()
    d['doy']=d['Date'].dt.dayofyear
    d['weekofyear']=d['Date'].dt.isocalendar().week.astype(int)
    d['quarter']=d['Date'].dt.quarter
    d['is_weekend']=(d['dow']>=5).astype(int)
    d['is_monthend']=d['Date'].dt.is_month_end.astype(int)
    d['is_payday']=(d['day'].isin([1,10,15,25])|d['Date'].dt.is_month_end).astype(int)
    d['sin_month']=np.sin(2*np.pi*d['month']/12)
    d['cos_month']=np.cos(2*np.pi*d['month']/12)
    d['sin_dow']=np.sin(2*np.pi*d['dow']/7)
    d['cos_dow']=np.cos(2*np.pi*d['dow']/7)
    d['sin_doy']=np.sin(2*np.pi*d['doy']/365)
    d['cos_doy']=np.cos(2*np.pi*d['doy']/365)
    d['dow_month']=d['dow']*12+d['month']
    return d

FEATS=['year','month','day','dow','doy','weekofyear','quarter','is_weekend','is_monthend','is_payday','sin_month','cos_month','sin_dow','cos_dow','sin_doy','cos_doy','dow_month']

eps=1.0
post2=sales[sales['year'].between(2020,2022)].copy()
btr=apply_stat_baseline(post2)
post2['lr'] = np.log((post2['Revenue']+eps)/(btr['pmd'].values+eps))
post2['lc'] = np.log((post2['COGS']+eps)/(btr['cmd'].values+eps))
for col in ['lr','lc']:
    q1,q3=post2[col].quantile([0.02,0.98]); post2[col]=post2[col].clip(q1,q3)

tr_f=make_features(post2[post2['year']<=2021].copy())
vl_f=make_features(val.copy())
tr_lr=post2.loc[post2['year']<=2021,'lr'].values
tr_lc=post2.loc[post2['year']<=2021,'lc'].values
vl_lr=post2.loc[post2['year']==2022,'lr'].values
vl_lc=post2.loc[post2['year']==2022,'lc'].values

ALPHA=0.5
ML_PREDS_R, ML_PREDS_C = [], []

import lightgbm as lgb
import xgboost as xgb

REG=dict(num_leaves=15,max_depth=3,min_child_samples=30,subsample=0.7,colsample_bytree=0.6,reg_alpha=1.0,reg_lambda=2.0,random_state=42,verbose=-1,n_jobs=-1)
lr=lgb.LGBMRegressor(n_estimators=300,learning_rate=0.03,**REG)
lr.fit(tr_f[FEATS],tr_lr,eval_set=[(vl_f[FEATS],vl_lr)],callbacks=[lgb.early_stopping(30,verbose=False)])
ML_PREDS_R.append(np.clip(bv['pmd'].values*np.exp(ALPHA*lr.predict(vl_f[FEATS])),0,None))

lc=lgb.LGBMRegressor(n_estimators=300,learning_rate=0.03,**REG)
lc.fit(tr_f[FEATS],tr_lc,eval_set=[(vl_f[FEATS],vl_lc)],callbacks=[lgb.early_stopping(30,verbose=False)])
ML_PREDS_C.append(np.clip(bv['cmd'].values*np.exp(ALPHA*lc.predict(vl_f[FEATS])),0,ML_PREDS_R[-1]*1.1))

xr=xgb.XGBRegressor(n_estimators=300,learning_rate=0.03,max_depth=3,min_child_weight=30,subsample=0.7,colsample_bytree=0.6,reg_alpha=1.0,reg_lambda=2.0,random_state=42,verbosity=0,n_jobs=-1)
xr.fit(tr_f[FEATS],tr_lr)
ML_PREDS_R.append(np.clip(bv['pmd'].values*np.exp(ALPHA*xr.predict(vl_f[FEATS])),0,None))

xc=xgb.XGBRegressor(n_estimators=300,learning_rate=0.03,max_depth=3,min_child_weight=30,subsample=0.7,colsample_bytree=0.6,reg_alpha=1.0,reg_lambda=2.0,random_state=42,verbosity=0,n_jobs=-1)
xc.fit(tr_f[FEATS],tr_lc)
ML_PREDS_C.append(np.clip(bv['cmd'].values*np.exp(ALPHA*xc.predict(vl_f[FEATS])),0,ML_PREDS_R[-1]*1.1))

try:
    from catboost import CatBoostRegressor
    cr=CatBoostRegressor(iterations=300,learning_rate=0.03,depth=3,l2_leaf_reg=2.0,subsample=0.7,random_seed=42,verbose=0)
    cr.fit(tr_f[FEATS],tr_lr)
    ML_PREDS_R.append(np.clip(bv['pmd'].values*np.exp(ALPHA*cr.predict(vl_f[FEATS])),0,None))
    cc=CatBoostRegressor(iterations=300,learning_rate=0.03,depth=3,l2_leaf_reg=2.0,subsample=0.7,random_seed=42,verbose=0)
    cc.fit(tr_f[FEATS],tr_lc)
    ML_PREDS_C.append(np.clip(bv['cmd'].values*np.exp(ALPHA*cc.predict(vl_f[FEATS])),0,ML_PREDS_R[-1]*1.1))
except:
    pass

ml_avg_r = np.mean(ML_PREDS_R, axis=0)
m = compute_metrics(val['Revenue'].values, ml_avg_r)
report_rows.append({'Mô hình': 'LightGBM/XGBoost/CatBoost', 'MAE': m[0], 'RMSE': m[1], 'R2': m[2], 'Ghi chú': 'Model chính'})

ens_r = np.clip(0.5 * ml_avg_r + 0.5 * bv['pmd'].values, 0, None)
m_ens = compute_metrics(val['Revenue'].values, ens_r)
report_rows.append({'Mô hình': 'Final ensemble/submission', 'MAE': m_ens[0], 'RMSE': m_ens[1], 'R2': m_ens[2], 'Ghi chú': 'Bản nộp Kaggle'})

In [31]:
pred_df=sub_form[['Date']].copy()
pred_df['year'] =pred_df['Date'].dt.year
pred_df['month']=pred_df['Date'].dt.month
pred_df['day']  =pred_df['Date'].dt.day
pred_df['dow']  =pred_df['Date'].dt.dayofweek

bp=apply_stat_baseline(pred_df)
pf=make_features(pred_df)

all_feat=make_features(post2.copy())

lr.fit(all_feat[FEATS],post2['lr'].values)
lc.fit(all_feat[FEATS],post2['lc'].values)
xr.fit(all_feat[FEATS],post2['lr'].values)
xc.fit(all_feat[FEATS],post2['lc'].values)

final_logr_preds = [lr.predict(pf[FEATS]), xr.predict(pf[FEATS])]
final_logc_preds = [lc.predict(pf[FEATS]), xc.predict(pf[FEATS])]

try:
    cr.fit(all_feat[FEATS],post2['lr'].values)
    cc.fit(all_feat[FEATS],post2['lc'].values)
    final_logr_preds.append(cr.predict(pf[FEATS]))
    final_logc_preds.append(cc.predict(pf[FEATS]))
except:
    pass

final_logr=np.mean(final_logr_preds, axis=0)
final_logc=np.mean(final_logc_preds, axis=0)

ml_final_r=np.clip(bp['pmd'].values*np.exp(ALPHA*final_logr),0,None)
ml_final_c=np.clip(bp['cmd'].values*np.exp(ALPHA*final_logc),0,ml_final_r*1.1)

final_r = np.clip(0.5 * ml_final_r + 0.5 * bp['pmd'].values, 0, None)
final_c = np.clip(0.5 * ml_final_c + 0.5 * bp['cmd'].values, 0, final_r*1.1)

sub=pd.DataFrame({'Date':pred_df['Date'].dt.strftime('%Y-%m-%d'),'Revenue':np.round(final_r,2),'COGS':np.round(final_c,2)})
sub.to_csv('submission/submission.csv',index=False)

In [32]:
df_report = pd.DataFrame(report_rows)
fmt = {'MAE': '{:,.0f}', 'RMSE': '{:,.0f}', 'R2': '{:.4f}'}
for col, f in fmt.items():
    df_report[col] = df_report[col].apply(lambda x: f.format(x))

def display_df(df):
    try:
        from IPython.display import display
        display(df)
    except:
        print(df.to_string(index=False))
display_df(df_report)

,Mô hình,MAE,RMSE,R2,Ghi chú
0,Seasonal naive,"837,704","1,161,819",0.5182,Baseline theo mùa vụ
1,Rolling average,"1,581,230","2,187,453",-0.7079,Baseline ổn định
2,LightGBM/XGBoost/CatBoost,"525,024","713,797",0.8181,Model chính
3,Final ensemble/submission,"527,587","714,133",0.8180,Bản nộp Kaggle
